# OBPOTF Simulations on Bivariate Bicycle Codes

This notebook runs BP+BP+OTF Monte Carlo simulations on the bivariate bicycle
codes from [Bravyi et al.](https://arxiv.org/abs/2308.07915):

| Code | Parameters | Distance |
|------|-----------|----------|
| BB-72 | [[72, 12, 6]] | 6 |
| BB-108 | [[108, 8, 10]] | 10 |
| BB-144 | [[144, 12, 12]] | 12 |
| BB-288 | [[288, 12, 18]] | 18 |

Pre-computed transfer matrices are loaded from `transfer_matrices/`.

The `css_code` class, `create_bivariate_bicycle_codes`, and `build_circuit`
functions below are adapted from the
[SlidingWindowDecoder](https://github.com/gongaa/SlidingWindowDecoder) package
(Copyright 2024 Anqi Gong, MIT License).

Reference: deMarti iOlius et al., [arXiv:2409.01440](https://arxiv.org/abs/2409.01440)

In [ ]:
import numpy as np
import scipy.io as sio
import stim
from functools import reduce
from scipy.sparse import identity, kron
from timeit import default_timer as timer

from BPOTF import OBPOTF, NoiseType, DemData
from BPOTF import __version__ as bpotf_version
from ldpc.ckt_noise.dem_matrices import detector_error_model_to_check_matrices
from ldpc.mod2 import row_echelon, rank

print(f"BPOTF version: v{bpotf_version}")


def kernel_with_pivots(mat):
    """Compute kernel, rank, and pivot columns of a binary matrix.

    ldpc.mod2.kernel only returns the kernel matrix. This wrapper also
    returns the rank and pivot column indices, which css_code needs.
    """
    transpose = mat.T
    m = transpose.shape[0]
    _, r, transform, pivot_cols = row_echelon(transpose)
    ker = transform[r:m]
    return ker, r, pivot_cols


# ---------------------------------------------------------------------------
# css_code class and create_bivariate_bicycle_codes
# Adapted from SlidingWindowDecoder (https://github.com/gongaa/SlidingWindowDecoder)
# Copyright 2024 Anqi Gong, MIT License
# ---------------------------------------------------------------------------

class css_code():
    """CSS quantum error-correcting code."""
    def __init__(self, hx=np.array([[]]), hz=np.array([[]]),
                 code_distance=np.nan, name=None, name_prefix="", check_css=False):
        self.hx = hx
        self.hz = hz
        self.lx = np.array([[]])
        self.lz = np.array([[]])
        self.N = np.nan
        self.K = np.nan
        self.D = code_distance
        self.L = np.nan
        self.Q = np.nan
        _, nx = self.hx.shape
        _, nz = self.hz.shape
        assert nx == nz, "hx and hz should have equal number of columns!"
        assert nx != 0, "number of variable nodes should not be zero!"
        if check_css:
            assert not np.any(hx @ hz.T % 2), "CSS constraint not satisfied"
        self.N = nx
        self.hx_perp, self.rank_hx, self.pivot_hx = kernel_with_pivots(hx)
        self.hz_perp, self.rank_hz, self.pivot_hz = kernel_with_pivots(hz)
        self.hx_basis = self.hx[self.pivot_hx]
        self.hz_basis = self.hz[self.pivot_hz]
        self.K = self.N - self.rank_hx - self.rank_hz
        self._compute_ldpc_params()
        self._compute_logicals()
        self.name = f"{name_prefix}_n{self.N}_k{self.K}" if name is None else name

    def _compute_ldpc_params(self):
        self.L = np.max([np.max(np.sum(self.hx, axis=0)), np.max(np.sum(self.hz, axis=0))]).astype(int)
        self.Q = np.max([np.max(np.sum(self.hx, axis=1)), np.max(np.sum(self.hz, axis=1))]).astype(int)

    def _compute_logicals(self):
        def compute_lz(ker_hx, im_hzT):
            log_stack = np.vstack([im_hzT, ker_hx])
            pivots = row_echelon(log_stack.T)[3]
            log_op_indices = [i for i in range(im_hzT.shape[0], log_stack.shape[0]) if i in pivots]
            return log_stack[log_op_indices]
        self.lx = compute_lz(self.hz_perp, self.hx_basis)
        self.lz = compute_lz(self.hx_perp, self.hz_basis)


def create_circulant_matrix(l, pows):
    """Create an l x l circulant matrix with 1s at the given power shifts."""
    h = np.zeros((l, l), dtype=int)
    for i in range(l):
        for c in pows:
            h[(i + c) % l, i] = 1
    return h


def create_bivariate_bicycle_codes(l, m, A_x_pows, A_y_pows, B_x_pows, B_y_pows,
                                   name=None, code_distance=np.nan):
    """Create a bivariate bicycle (BB) CSS code over Z_l x Z_m.

    Pass code_distance if known to skip the expensive exact distance computation.
    """
    S_l = create_circulant_matrix(l, [-1])
    S_m = create_circulant_matrix(m, [-1])
    x = kron(S_l, identity(m, dtype=int))
    y = kron(identity(l, dtype=int), S_m)
    A_list = [x**p for p in A_x_pows] + [y**p for p in A_y_pows]
    B_list = [y**p for p in B_y_pows] + [x**p for p in B_x_pows]
    A = reduce(lambda x, y: x + y, A_list).toarray()
    B = reduce(lambda x, y: x + y, B_list).toarray()
    hx = np.hstack((A, B))
    hz = np.hstack((B.T, A.T))
    return css_code(hx, hz, code_distance=code_distance, name=name,
                    name_prefix="BB", check_css=True), A_list, B_list


# ---------------------------------------------------------------------------
# build_circuit: stim circuit construction for BB codes
# Adapted from SlidingWindowDecoder (https://github.com/gongaa/SlidingWindowDecoder)
# Copyright 2024 Anqi Gong, MIT License
# ---------------------------------------------------------------------------

def build_circuit(code, A_list, B_list, p, num_repeat, z_basis=True, use_both=False, HZH=False):
    """Build a stim circuit for a bivariate bicycle code with circuit-level noise."""
    n = code.N
    a1, a2, a3 = A_list
    b1, b2, b3 = B_list

    def nnz(m):
        a, b = m.nonzero()
        return b[np.argsort(a)]

    A1, A2, A3 = nnz(a1), nnz(a2), nnz(a3)
    B1, B2, B3 = nnz(b1), nnz(b2), nnz(b3)
    A1_T, A2_T, A3_T = nnz(a1.T), nnz(a2.T), nnz(a3.T)
    B1_T, B2_T, B3_T = nnz(b1.T), nnz(b2.T), nnz(b3.T)

    X_check_offset = 0
    L_data_offset = n // 2
    R_data_offset = n
    Z_check_offset = 3 * n // 2

    p_dep = p
    p_res = p
    p_mea = p
    p_rnd = p

    det_str = ""
    for i in range(n // 2):
        det_str += f"DETECTOR rec[{-n // 2 + i}]\n"
    detector_circuit = stim.Circuit(det_str)

    det_rep_str = ""
    for i in range(n // 2):
        det_rep_str += f"DETECTOR rec[{-n // 2 + i}] rec[{-n - n // 2 + i}]\n"
    detector_repeat_circuit = stim.Circuit(det_rep_str)

    def append_blocks(circuit, repeat=False):
        if repeat:
            for i in range(n // 2):
                circuit.append("X_ERROR", Z_check_offset + i, p_res)
                if HZH:
                    circuit.append("X_ERROR", X_check_offset + i, p_res)
                    circuit.append("H", [X_check_offset + i])
                    circuit.append("DEPOLARIZE1", X_check_offset + i, p_dep)
                else:
                    circuit.append("Z_ERROR", X_check_offset + i, p_res)
                circuit.append("DEPOLARIZE1", R_data_offset + i, p_rnd)
        else:
            for i in range(n // 2):
                circuit.append("H", [X_check_offset + i])
                if HZH:
                    circuit.append("DEPOLARIZE1", X_check_offset + i, p_dep)
        for i in range(n // 2):
            circuit.append("CNOT", [R_data_offset + A1_T[i], Z_check_offset + i])
            circuit.append("DEPOLARIZE2", [R_data_offset + A1_T[i], Z_check_offset + i], p_dep)
            circuit.append("DEPOLARIZE1", L_data_offset + i, p_rnd)
        circuit.append("TICK")
        for i in range(n // 2):
            circuit.append("CNOT", [X_check_offset + i, L_data_offset + A2[i]])
            circuit.append("DEPOLARIZE2", [X_check_offset + i, L_data_offset + A2[i]], p_dep)
            circuit.append("CNOT", [R_data_offset + A3_T[i], Z_check_offset + i])
            circuit.append("DEPOLARIZE2", [R_data_offset + A3_T[i], Z_check_offset + i], p_dep)
        circuit.append("TICK")
        for i in range(n // 2):
            circuit.append("CNOT", [X_check_offset + i, R_data_offset + B2[i]])
            circuit.append("DEPOLARIZE2", [X_check_offset + i, R_data_offset + B2[i]], p_dep)
            circuit.append("CNOT", [L_data_offset + B1_T[i], Z_check_offset + i])
            circuit.append("DEPOLARIZE2", [L_data_offset + B1_T[i], Z_check_offset + i], p_dep)
        circuit.append("TICK")
        for i in range(n // 2):
            circuit.append("CNOT", [X_check_offset + i, R_data_offset + B1[i]])
            circuit.append("DEPOLARIZE2", [X_check_offset + i, R_data_offset + B1[i]], p_dep)
            circuit.append("CNOT", [L_data_offset + B2_T[i], Z_check_offset + i])
            circuit.append("DEPOLARIZE2", [L_data_offset + B2_T[i], Z_check_offset + i], p_dep)
        circuit.append("TICK")
        for i in range(n // 2):
            circuit.append("CNOT", [X_check_offset + i, R_data_offset + B3[i]])
            circuit.append("DEPOLARIZE2", [X_check_offset + i, R_data_offset + B3[i]], p_dep)
            circuit.append("CNOT", [L_data_offset + B3_T[i], Z_check_offset + i])
            circuit.append("DEPOLARIZE2", [L_data_offset + B3_T[i], Z_check_offset + i], p_dep)
        circuit.append("TICK")
        for i in range(n // 2):
            circuit.append("CNOT", [X_check_offset + i, L_data_offset + A1[i]])
            circuit.append("DEPOLARIZE2", [X_check_offset + i, L_data_offset + A1[i]], p_dep)
            circuit.append("CNOT", [R_data_offset + A2_T[i], Z_check_offset + i])
            circuit.append("DEPOLARIZE2", [R_data_offset + A2_T[i], Z_check_offset + i], p_dep)
        circuit.append("TICK")
        for i in range(n // 2):
            circuit.append("CNOT", [X_check_offset + i, L_data_offset + A3[i]])
            circuit.append("DEPOLARIZE2", [X_check_offset + i, L_data_offset + A3[i]], p_dep)
            circuit.append("X_ERROR", Z_check_offset + i, p_mea)
            circuit.append("MR", [Z_check_offset + i])
        if z_basis:
            circuit += (detector_repeat_circuit if repeat else detector_circuit)
        elif use_both and repeat:
            circuit += detector_repeat_circuit
        circuit.append("TICK")
        for i in range(n // 2):
            if HZH:
                circuit.append("H", [X_check_offset + i])
                circuit.append("DEPOLARIZE1", X_check_offset + i, p_dep)
                circuit.append("X_ERROR", X_check_offset + i, p_mea)
                circuit.append("MR", [X_check_offset + i])
            else:
                circuit.append("Z_ERROR", X_check_offset + i, p_mea)
                circuit.append("MRX", [X_check_offset + i])
        if not z_basis:
            circuit += (detector_repeat_circuit if repeat else detector_circuit)
        elif use_both and repeat:
            circuit += detector_repeat_circuit
        circuit.append("TICK")

    circuit = stim.Circuit()
    for i in range(n // 2):
        circuit.append("R", X_check_offset + i)
        circuit.append("R", Z_check_offset + i)
        circuit.append("X_ERROR", X_check_offset + i, p_res)
        circuit.append("X_ERROR", Z_check_offset + i, p_res)
    for i in range(n):
        circuit.append("R" if z_basis else "RX", L_data_offset + i)
        circuit.append("X_ERROR" if z_basis else "Z_ERROR", L_data_offset + i, p_res)
    circuit.append("TICK")
    append_blocks(circuit, repeat=False)
    rep_circuit = stim.Circuit()
    append_blocks(rep_circuit, repeat=True)
    circuit += (num_repeat - 1) * rep_circuit
    for i in range(n):
        circuit.append("M" if z_basis else "MX", L_data_offset + i)
    pcm = code.hz if z_basis else code.hx
    logical_pcm = code.lz if z_basis else code.lx
    stab_str = ""
    for i, s in enumerate(pcm):
        nnz_idx = np.nonzero(s)[0]
        det = "DETECTOR"
        for ind in nnz_idx:
            det += f" rec[{-n + ind}]"
        det += f" rec[{-n - n + i}]" if z_basis else f" rec[{-n - n // 2 + i}]"
        stab_str += det + "\n"
    circuit += stim.Circuit(stab_str)
    log_str = ""
    for i, l in enumerate(logical_pcm):
        nnz_idx = np.nonzero(l)[0]
        obs = f"OBSERVABLE_INCLUDE({i})"
        for ind in nnz_idx:
            obs += f" rec[{-n + ind}]"
        log_str += obs + "\n"
    circuit += stim.Circuit(log_str)
    return circuit

## 1. Define the BB code configurations

In [ ]:
BB_CODES = {
    72: {
        "l": 6, "m": 6,
        "A_x_pows": [3], "A_y_pows": [1, 2],
        "B_x_pows": [1, 2], "B_y_pows": [3],
        "d": 6,
        "transfer_file": "transfer_matrices/PhenoTransf72Test.mat",
    },
    108: {
        "l": 9, "m": 6,
        "A_x_pows": [3], "A_y_pows": [1, 2],
        "B_x_pows": [1, 2], "B_y_pows": [3],
        "d": 10,
        "transfer_file": "transfer_matrices/PhenoTransf108Test.mat",
    },
    144: {
        "l": 12, "m": 6,
        "A_x_pows": [3], "A_y_pows": [1, 2],
        "B_x_pows": [1, 2], "B_y_pows": [3],
        "d": 12,
        "transfer_file": "transfer_matrices/PhenoTransf144Test.mat",
    },
    288: {
        "l": 12, "m": 12,
        "A_x_pows": [3], "A_y_pows": [2, 7],
        "B_x_pows": [1, 2], "B_y_pows": [3],
        "d": 18,
        "transfer_file": "transfer_matrices/PhenoTransf288Test.mat",
    },
}

## 2. Helper: build decoder for a BB code

In [ ]:
def build_bb_decoder(bb_type, p, bp_iterations=None, decimation=1e-9):
    """Build a circuit, DEM, and OBPOTF decoder for a given BB code.

    Parameters
    ----------
    bb_type : int
        Code block length (72, 108, 144, or 288).
    p : float
        Physical error rate.
    bp_iterations : list of 3 ints or None
        Max BP iterations for [DEM stage, phenomenological stage, OTF stage].
    decimation : float
        Decimation parameter for the OTF stage. Controls the initial LLR
        value assigned to columns that are NOT selected by the OTF
        (Kruskal) algorithm. A value of 0 means unselected columns are
        completely suppressed; larger values allow a small residual
        probability for unselected columns during the final BP round.

    Returns
    -------
    decoder : OBPOTF
    circuit : stim.Circuit
    bm : check matrix bundle
    """
    cfg = BB_CODES[bb_type]
    d = cfg["d"]

    # Build code and circuit
    # Pass code_distance to skip the expensive exact distance computation
    code, A_list, B_list = create_bivariate_bicycle_codes(
        cfg["l"], cfg["m"],
        cfg["A_x_pows"], cfg["A_y_pows"],
        cfg["B_x_pows"], cfg["B_y_pows"],
        code_distance=d,
    )
    circuit = build_circuit(code, A_list, B_list, p=p, num_repeat=d, z_basis=True)
    dem = circuit.detector_error_model()
    bm = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)

    # Load pre-computed transfer matrix and phenomenological matrices
    mat_data = sio.loadmat(cfg["transfer_file"])
    transfer_mat = mat_data["transfMatDEMtoPheno"]
    if hasattr(transfer_mat, 'toarray'):
        transfer_mat = transfer_mat.toarray('F')
    phen_check = mat_data["dem_pheno"]
    phen_obs = mat_data["obsphen"]

    # Build DemData
    dem_data = DemData()
    dem_data.priors = bm.priors
    dem_data.obs_matrix = bm.observables_matrix.toarray('F').astype(np.uint8)
    dem_data.transfer_matrix = transfer_mat.astype(np.uint8)
    dem_data.phen_check_matrix = phen_check.astype(np.uint8)
    dem_data.phen_obs_matrix = phen_obs.astype(np.uint8)

    # BP iteration counts
    if bp_iterations is not None:
        bp_iters = np.array(bp_iterations, dtype=np.int32)
    else:
        bp_iters = None

    decoder = OBPOTF(
        bm.check_matrix,
        p,
        NoiseType.E_CLN,
        ps_ext_dem_data=dem_data,
        po_ext_bp_iters=bp_iters,
        decimation=decimation,
    )

    return decoder, circuit, bm

## 3. Run simulations

In [ ]:
# Simulation parameters
bb_types = [72, 108, 144, 288]
p_values = [1e-3, 2e-3, 3e-3]
NMC = 5000  # Monte Carlo shots per configuration
bp_iterations = [100, 400, 100]  # [DEM stage, pheno stage, OTF stage]

# Decimation: controls the initial LLR assigned to columns not selected by OTF.
# A value of 1e-9 assigns a near-zero probability to unselected columns,
# effectively suppressing them while keeping numerical stability.
decimation = 1e-9

results = {}

for bb_type in bb_types:
    d = BB_CODES[bb_type]["d"]
    results[bb_type] = {}

    for p in p_values:
        print(f"\n{'='*60}")
        print(f"BB-{bb_type} [[{bb_type}, *, {d}]] @ p = {p}")
        print(f"{'='*60}")

        decoder, circuit, bm = build_bb_decoder(bb_type, p, bp_iterations, decimation)
        sampler = circuit.compile_detector_sampler()

        num_errors = 0
        total_time = 0.0

        detection_events, observable_flips = sampler.sample(
            NMC, separate_observables=True
        )

        start = timer()
        for i in range(NMC):
            prediction = decoder.decode(detection_events[i].astype(np.uint8))
            if not np.all(prediction == observable_flips[i]):
                num_errors += 1
        total_time = timer() - start

        error_rate = num_errors / NMC
        error_rate_per_round = error_rate / d
        avg_time_per_shot = total_time / NMC

        results[bb_type][p] = {
            "shots": NMC,
            "errors": num_errors,
            "error_rate": error_rate,
            "error_rate_per_round": error_rate_per_round,
            "total_time": total_time,
            "avg_time_per_shot": avg_time_per_shot,
        }

        print(f"  Shots: {NMC}, Errors: {num_errors}")
        print(f"  p(e): {error_rate:.6f}")
        print(f"  p(e)/d: {error_rate_per_round:.6f}")
        print(f"  Total time: {total_time:.2f}s, Avg: {avg_time_per_shot*1000:.2f} ms/shot")

## 4. Results table

In [ ]:
print(f"{'Code':<10} {'d':<5} {'p':<10} {'shots':<8} {'errors':<8} {'p(e)':<12} {'p(e)/d':<12} {'ms/shot':<10}")
print("-" * 80)

for bb_type in bb_types:
    d = BB_CODES[bb_type]["d"]
    for p in p_values:
        r = results[bb_type][p]
        print(
            f"BB-{bb_type:<6} {d:<5} {p:<10.4f} {r['shots']:<8} {r['errors']:<8} "
            f"{r['error_rate']:<12.6f} {r['error_rate_per_round']:<12.8f} "
            f"{r['avg_time_per_shot']*1000:<10.2f}"
        )

## 5. Plot logical error rate per round

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))

for bb_type in bb_types:
    d = BB_CODES[bb_type]["d"]
    ps = sorted(results[bb_type].keys())
    rates = [results[bb_type][p]["error_rate_per_round"] for p in ps]
    # Only plot points with at least 1 error
    valid = [(pp, rr) for pp, rr in zip(ps, rates) if rr > 0]
    if valid:
        vp, vr = zip(*valid)
        ax.plot(vp, vr, "o-", label=f"[[{bb_type}, *, {d}]]")

ax.set_xlabel("Physical error rate")
ax.set_ylabel("Logical error rate per round")
ax.set_xscale("log")
ax.set_yscale("log")
ax.legend()
ax.set_title("BP+BP+OTF -- Bivariate Bicycle Codes")
ax.grid(True, which="both", ls="--", alpha=0.5)
plt.tight_layout()
plt.show()